In [ ]:
import pandas as pd
from pass_pclr.defines import ECHONEXT_TARGETS
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
target_cols = list(ECHONEXT_TARGETS.keys())
subset_labels = []
for suffix in ["", "-32k", "-16k", "-8k", "-4k", "-2k", "-1k", "-512", "-256"]:
    df = pd.read_csv(f"/opt/gpudata/ecg/echonext{suffix}/EchoNext_metadata_100k.csv")
    df = df[df["split"] == "train"]
    df = df.rename(columns={v: k for k, v in ECHONEXT_TARGETS.items()})
    df = df[target_cols].reset_index(drop=True)
    df["subset"] = len(df)
    subset_labels.append(df)

In [ ]:
df = pd.concat(subset_labels, ignore_index=True)

In [ ]:
import math

def best_square_subplot_layout(n, fig_width=16, fig_height=9):
    """
    Determine rows, cols, and subplot size (in inches) for n square subplots
    that maximize subplot size within a fixed figure size.
    """
    best = {
        "rows": None,
        "cols": None,
        "subplot_size": 0.0
    }

    # rows can range from 1..n
    for rows in range(1, n + 1):
        cols = math.ceil(n / rows)

        # max square size constrained by width and height
        size_w = fig_width / cols
        size_h = fig_height / rows
        square_size = min(size_w, size_h)

        if square_size > best["subplot_size"]:
            best.update(
                rows=rows,
                cols=cols,
                subplot_size=square_size
            )

    return best


In [ ]:
layout = best_square_subplot_layout(len(target_cols))
fig, axes = plt.subplots(layout["rows"], layout["cols"], figsize=(24, 13.5))
for label, ax in zip(target_cols, axes.flat):
    sns.barplot(df, x="subset", log_scale=0, y=label, errorbar=None, ax=ax)
    ax.set_ylabel("Label Prevalence")
    ax.set_title(label)
fig.tight_layout()